In [27]:
import os 
from datetime import datetime 
import pandas as pd 
import numpy as np 
import json

In [28]:
def allocate(order_size, venues, lam_over, lam_under, theta_queue):
    step = 100
    splits = [[]]
    for v in range(len(venues)):
        new_splits = []
        for alloc in splits:
            used = sum(alloc)
            max_v = min(order_size - used, venues[v]['ask_size'])
            for q in range(0, max_v + 1, step):
                new_splits.append(alloc + [q])
        splits = new_splits
    best_cost = float('inf')
    best_split = []
    for alloc in splits:
        if sum(alloc) != order_size:
            continue
        cost = compute_cost(alloc, venues, order_size, lam_over, lam_under, theta_queue)
        if cost < best_cost:
            best_cost = cost
            best_split = alloc
    return best_split, best_cost

def compute_cost(split, venues, order_size, lam_over, lam_under, theta_queue):
    executed = 0
    cash_spent = 0.0
    for i in range(len(venues)):
        exe = min(split[i], venues[i]['ask_size'])
        executed += exe
        cash_spent += exe * (venues[i]['ask'] + venues[i].get('fee', 0))
        maker_rebate = max(split[i] - exe, 0) * venues[i].get('rebate', 0)
        cash_spent -= maker_rebate
    underfill = max(order_size - executed, 0)
    overfill = max(executed - order_size, 0)
    risk_pen = theta_queue * (underfill + overfill)
    cost_pen = lam_under * underfill + lam_over * overfill
    return cash_spent + risk_pen + cost_pen

In [29]:
df = pd.read_csv("l1_day.csv")
print(df.columns)
print(df.shape)
df.head()

Index(['ts_recv', 'ts_event', 'rtype', 'publisher_id', 'instrument_id',
       'action', 'side', 'depth', 'price', 'size', 'flags', 'ts_in_delta',
       'sequence', 'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00',
       'bid_ct_00', 'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01',
       'ask_sz_01', 'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02',
       'bid_sz_02', 'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03',
       'ask_px_03', 'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03',
       'bid_px_04', 'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04',
       'ask_ct_04', 'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05',
       'bid_ct_05', 'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06',
       'ask_sz_06', 'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07',
       'bid_sz_07', 'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08',
       'ask_px_08', 'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08',
       'bid_px_09', 'ask_px_09', 'bid_sz_09', 'ask_sz_09', 'bi

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491911683Z,10,2,38,T,A,0,222.81,190,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
1,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491911683Z,10,2,38,T,A,0,222.81,10,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
2,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491911683Z,10,2,38,T,A,0,222.81,100,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
3,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491911683Z,10,2,38,T,A,0,222.81,121,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
4,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491912675Z,10,2,38,A,N,0,222.83,10,...,100,3,1,222.71,222.94,1163,317,7,5,AAPL


In [30]:
df.iloc[0].to_csv('test.txt', header=True)

In [31]:
df.iloc[0]['ask_px_00'], df.iloc[0]['ask_sz_00'], df.iloc[0]['bid_px_00'], df.iloc[0]['bid_sz_00']

(np.float64(222.83), np.int64(36), np.float64(222.81), np.int64(421))

In [32]:
df = pd.read_csv("l1_day.csv")
# order df by ['ts_event'] and ['publisher_id']
df = df.sort_values(by=['ts_event', 'publisher_id'])
print(f"Before dropping duplicates: {df.shape}")
# drop duplicates keeping first message per publisher_id for each ts_event
df = df.drop_duplicates(subset=['ts_event', 'publisher_id'])
print(f"After dropping duplicates: {df.shape}")
df.head()

Before dropping duplicates: (59999, 74)
After dropping duplicates: (54537, 74)


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491911683Z,10,2,38,T,A,0,222.81,190,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
4,2024-08-01T13:36:32.492082308Z,2024-08-01T13:36:32.491912675Z,10,2,38,A,N,0,222.83,10,...,100,3,1,222.71,222.94,1163,317,7,5,AAPL
5,2024-08-01T13:36:32.492093947Z,2024-08-01T13:36:32.491927912Z,10,2,38,A,A,0,222.83,100,...,100,3,1,222.71,222.94,1163,317,7,5,AAPL
6,2024-08-01T13:36:32.492105645Z,2024-08-01T13:36:32.491939846Z,10,2,38,A,A,0,222.82,87,...,219,3,3,222.71,222.93,1163,100,7,1,AAPL
7,2024-08-01T13:36:32.492110610Z,2024-08-01T13:36:32.491944432Z,10,2,38,A,B,1,222.79,100,...,219,3,3,222.71,222.93,1163,100,7,1,AAPL


In [33]:
# Convert the timestamp string to datetime object with nanosecond precision
timestamp_str = df.iloc[0]['ts_event']
timestamp = pd.to_datetime(timestamp_str, format='%Y-%m-%dT%H:%M:%S.%fZ')
print(f"Original string: {timestamp_str}")
print(f"Converted datetime: {timestamp}")
df['ts_event'] = pd.to_datetime(df['ts_event'], format='%Y-%m-%dT%H:%M:%S.%fZ')
df.head()

Original string: 2024-08-01T13:36:32.491911683Z
Converted datetime: 2024-08-01 13:36:32.491911683


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-08-01T13:36:32.492082308Z,2024-08-01 13:36:32.491911683,10,2,38,T,A,0,222.81,190,...,100,4,1,222.72,222.94,219,317,3,5,AAPL
4,2024-08-01T13:36:32.492082308Z,2024-08-01 13:36:32.491912675,10,2,38,A,N,0,222.83,10,...,100,3,1,222.71,222.94,1163,317,7,5,AAPL
5,2024-08-01T13:36:32.492093947Z,2024-08-01 13:36:32.491927912,10,2,38,A,A,0,222.83,100,...,100,3,1,222.71,222.94,1163,317,7,5,AAPL
6,2024-08-01T13:36:32.492105645Z,2024-08-01 13:36:32.491939846,10,2,38,A,A,0,222.82,87,...,219,3,3,222.71,222.93,1163,100,7,1,AAPL
7,2024-08-01T13:36:32.492110610Z,2024-08-01 13:36:32.491944432,10,2,38,A,B,1,222.79,100,...,219,3,3,222.71,222.93,1163,100,7,1,AAPL


In [34]:
df = df[['ts_event', 
         'publisher_id', 
         'ask_px_00', 'ask_sz_00', 'bid_px_00', 'bid_sz_00',
         'ask_px_01', 'ask_sz_01', 'bid_px_01', 'bid_sz_01',
         'ask_px_02', 'ask_sz_02', 'bid_px_02', 'bid_sz_02',
         'ask_px_03', 'ask_sz_03', 'bid_px_03', 'bid_sz_03',
         'ask_px_04', 'ask_sz_04', 'bid_px_04', 'bid_sz_04',
         'ask_px_05', 'ask_sz_05', 'bid_px_05', 'bid_sz_05',
         'ask_px_06', 'ask_sz_06', 'bid_px_06', 'bid_sz_06',
         'ask_px_07', 'ask_sz_07', 'bid_px_07', 'bid_sz_07',
         'ask_px_08', 'ask_sz_08', 'bid_px_08', 'bid_sz_08',
         'ask_px_09', 'ask_sz_09', 'bid_px_09', 'bid_sz_09',
         ]]
# index by ts_event
df = df.set_index('ts_event')

In [35]:
df.head()

,publisher_id,ask_px_00,ask_sz_00,bid_px_00,bid_sz_00,ask_px_01,ask_sz_01,bid_px_01,bid_sz_01,ask_px_02,...,bid_px_07,bid_sz_07,ask_px_08,ask_sz_08,bid_px_08,bid_sz_08,ask_px_09,ask_sz_09,bid_px_09,bid_sz_09
ts_event,,,,,,,,,,,,,,,,,,,,,
2024-08-01 13:36:32.491911683,2,222.83,36,222.81,421,222.84,300,222.80,100,222.86,...,222.74,410,222.93,100,222.73,215,222.94,317,222.72,219
2024-08-01 13:36:32.491912675,2,222.83,46,222.80,100,222.84,300,222.79,135,222.86,...,222.73,215,222.93,100,222.72,219,222.94,317,222.71,1163
2024-08-01 13:36:32.491927912,2,222.83,146,222.80,100,222.84,300,222.79,135,222.86,...,222.73,215,222.93,100,222.72,219,222.94,317,222.71,1163
2024-08-01 13:36:32.491939846,2,222.82,87,222.80,100,222.83,146,222.79,135,222.84,...,222.73,215,222.92,219,222.72,219,222.93,100,222.71,1163
2024-08-01 13:36:32.491944432,2,222.82,87,222.80,100,222.83,146,222.79,235,222.84,...,222.73,215,222.92,219,222.72,219,222.93,100,222.71,1163


In [36]:
# unique publisher_id
publisher_ids = pd.read_csv('l1_day.csv')['publisher_id'].unique()
print(publisher_ids)

[2]


In [37]:
from dataclasses import dataclass
@dataclass 
class Venue:
    ask: float
    ask_sz: int
    ask_01: float
    ask_01_sz: int
    ask_02: float
    ask_02_sz: int
    ask_03: float
    ask_03_sz: int
    ask_04: float
    ask_04_sz: int
    ask_05: float
    ask_05_sz: int
    ask_06: float
    ask_06_sz: int
    ask_07: float
    ask_07_sz: int
    ask_08: float
    ask_08_sz: int
    ask_09: float
    ask_09_sz: int
    fee: float = 0.0000
    rebate: float = 0.0030 

In [38]:
# make df indexed by ts_event, and only contain columns 
# venues, where venues is a list of Venue objects 

# Group by ts_event and create a list of Venue objects for each timestamp
df['venue'] = df.apply(lambda row: Venue(
    ask=row['ask_px_00'],
    ask_sz=row['ask_sz_00'],
    ask_01=row['ask_px_01'],
    ask_01_sz=row['ask_sz_01'],
    ask_02=row['ask_px_02'],
    ask_02_sz=row['ask_sz_02'],
    ask_03=row['ask_px_03'],
    ask_03_sz=row['ask_sz_03'],
    ask_04=row['ask_px_04'],
    ask_04_sz=row['ask_sz_04'],
    ask_05=row['ask_px_05'],
    ask_05_sz=row['ask_sz_05'],
    ask_06=row['ask_px_06'],
    ask_06_sz=row['ask_sz_06'],
    ask_07=row['ask_px_07'],
    ask_07_sz=row['ask_sz_07'],
    ask_08=row['ask_px_08'],
    ask_08_sz=row['ask_sz_08'],
    ask_09=row['ask_px_09'],
    ask_09_sz=row['ask_sz_09'],
    fee=0.0000,
    rebate=0.0030
), axis=1)

# Group by ts_event and aggregate venues into a list
df = df.groupby('ts_event')['venue'].apply(list).reset_index()
df = df.set_index('ts_event')
df.head()

,venue
ts_event,
2024-08-01 13:36:32.491911683,"[Venue(ask=np.float64(222.83), ask_sz=np.float..."
2024-08-01 13:36:32.491912675,"[Venue(ask=np.float64(222.83), ask_sz=np.float..."
2024-08-01 13:36:32.491927912,"[Venue(ask=np.float64(222.83), ask_sz=np.float..."
2024-08-01 13:36:32.491939846,"[Venue(ask=np.float64(222.82), ask_sz=np.float..."
2024-08-01 13:36:32.491944432,"[Venue(ask=np.float64(222.82), ask_sz=np.float..."


In [44]:
df['venue'][0][0]

/var/folders/ky/cz3py0bj5w1c3ln1rt5ncs200000gn/T/ipykernel_51007/2740448313.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df['venue'][0][0]


Venue(ask=np.float64(222.83), ask_sz=np.float64(36.0), ask_01=np.float64(222.84), ask_01_sz=np.float64(300.0), ask_02=np.float64(222.86), ask_02_sz=np.float64(100.0), ask_03=np.float64(222.87), ask_03_sz=np.float64(110.0), ask_04=np.float64(222.89), ask_04_sz=np.float64(100.0), ask_05=np.float64(222.9), ask_05_sz=np.float64(300.0), ask_06=np.float64(222.91), ask_06_sz=np.float64(300.0), ask_07=np.float64(222.92), ask_07_sz=np.float64(219.0), ask_08=np.float64(222.93), ask_08_sz=np.float64(100.0), ask_09=np.float64(222.94), ask_09_sz=np.float64(317.0), fee=0.0, rebate=0.003)